# Lumped Conceptual Model

- please change the Path to the directory where you stored the case study data

In [ ]:
import os

Comp = "F:/01Algorithms/Hydrology/HAPI/Examples"
os.chdir(Comp)

### Import Modules

In [ ]:
import datetime as dt

import Hapi.rrm.hbv_bergestrom92 as HBVLumped
import Hapi.sm.performancecriteria as PC
import matplotlib.pyplot as plt
from Hapi.catchment import Catchment
from Hapi.rrm.routing import Routing
from Hapi.run import Run

### Paths

In [ ]:
Parameterpath = Comp + "/data/lumped/Coello_Lumped2021-03-08_muskingum.txt"
MeteoDataPath = Comp + "/data/lumped/meteo_data-MSWEP.csv"
Path = Comp + "/data/lumped/"

### Meteorological data

In [ ]:
start = "2009-01-01"
end = "2011-12-31"
name = "Coello"
Coello = Catchment(name, start, end)
Coello.readLumpedInputs(MeteoDataPath)

### Lumped model

In [ ]:
# catchment area
AreaCoeff = 1530
# [Snow pack, Soil moisture, Upper zone, Lower Zone, Water content]
InitialCond = [0, 10, 10, 10, 0]

Coello.readLumpedModel(HBVLumped, AreaCoeff, InitialCond)

### Model Parameters

In [ ]:
Snow = 0  # no snow subroutine
Coello.readParameters(Parameterpath, Snow)

In [ ]:
Coello.parameters

### Observed flow

In [ ]:
Coello.readDischargeGauges(Path + "Qout_c.csv", fmt="%Y-%m-%d")

- the discharge data should be stored in the text file with the date stored in the first column and the discharge values in the second column

### Routing

In [ ]:
RoutingFn = Routing.muskingum_v
Route = 1

### Run The Model

In [ ]:
Run.run_lumped(Coello, Route, RoutingFn)

### Calculate performance criteria

In [ ]:
Metrics = {}

Qobs = Coello.QGauges['q']

Metrics['RMSE'] = PC.RMSE(Qobs, Coello.Qsim['q'])
Metrics['NSE'] = PC.NSE(Qobs, Coello.Qsim['q'])
Metrics['NSEhf'] = PC.NSEHF(Qobs, Coello.Qsim['q'])
Metrics['KGE'] = PC.KGE(Qobs, Coello.Qsim['q'])
Metrics['WB'] = PC.WB(Qobs, Coello.Qsim['q'])

print("RMSE= " + str(round(Metrics['RMSE'], 2)))
print("NSE= " + str(round(Metrics['NSE'], 2)))
print("NSEhf= " + str(round(Metrics['NSEhf'], 2)))
print("KGE= " + str(round(Metrics['KGE'], 2)))
print("WB= " + str(round(Metrics['WB'], 2)))

### Plot Hydrograph

In [ ]:
gaugei = 0
plotstart = "2009-01-01"
plotend = "2011-12-31"
Coello.plotHydrograph(plotstart, plotend, gaugei, Title="Lumped Model")

### Save Results

In [ ]:
StartDate = "2009-01-01"
EndDate = "2010-04-20"

Path = SaveTo + "Results-Lumped-Model" + str(dt.datetime.now())[0:10] + ".txt"
Coello.saveResults(Result=5, StartDate=StartDate, EndDate=EndDate, Path=Path)